[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/adv-11-rlhf-on-six-replies.ipynb)

# Advanced Discussion 11 — policy gradient, KL penalty, PPO clipping and DPO on a toy policy

REINFORCE with and without a baseline, the KL leash swept over beta against the closed-form optimum, PPO clipping, and DPO from preference pairs.

**Optional advanced-discussion lab** for Generative AI and Large Language Models with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Pure PyTorch on a six-reply toy; runs in about a minute.

In [ ]:
import numpy as np, torch, torch.nn.functional as F
torch.manual_seed(0); np.random.seed(0)

## the toy: a "model" chooses one of 6 candidate replies; a hidden reward says how good each one is

In [ ]:
r_true = torch.tensor([0.0, 0.2, 0.5, 1.0, 0.4, 0.8])            # the reward model's score for each candidate reply
ref_logits = torch.tensor([1.5, 1.0, 0.8, -0.5, 0.6, 0.0])       # the SFT (reference) policy: prefers the low-reward replies
pi_ref = F.softmax(ref_logits, 0)
print("reference policy:", pi_ref.numpy().round(3), "expected reward %.3f" % (pi_ref @ r_true).item())
kl = lambda p, q: (p * (p.clamp_min(1e-12).log() - q.clamp_min(1e-12).log())).sum()

## 1. REINFORCE: sample a reply, weight its log-probability by (reward - baseline)

In [ ]:
def reinforce(beta=0.0, steps=800, batch=32, lr=0.1, baseline=True, seed=0):
    g = torch.Generator().manual_seed(seed); theta = ref_logits.clone().requires_grad_(True); opt = torch.optim.SGD([theta], lr=lr); grad_vars = []
    for _ in range(steps):
        pi = F.softmax(theta, 0); a = torch.multinomial(pi.detach(), batch, replacement=True, generator=g)
        logp = pi.log()[a]
        shaped = r_true[a] - beta * (logp.detach() - pi_ref.log()[a])               # reward minus KL penalty, per sample
        adv = shaped - (shaped.mean() if baseline else 0.0)
        loss = -(adv * logp).mean(); opt.zero_grad(); loss.backward(); opt.step()
    return F.softmax(theta.detach(), 0)
def grad_variance(baseline, trials=2000, batch=8):
    theta = ref_logits.clone().requires_grad_(True); gs = []
    for t in range(trials):
        pi = F.softmax(theta, 0); a = torch.multinomial(pi.detach(), batch, replacement=True); logp = pi.log()[a]
        adv = r_true[a] - (r_true[a].mean() if baseline else 0.0)
        g_, = torch.autograd.grad(-(adv * logp).mean(), theta); gs.append(g_)
    return torch.stack(gs).var(0).sum().item()
print("\ngradient variance (sum over parameters, batch of 8): no baseline %.4f | with baseline %.4f" % (grad_variance(False), grad_variance(True)))

## 2. the KL penalty: the price of staying close to the reference

In [ ]:
print("\nbeta   expected reward   KL(pi || ref)   P(best reply)   entropy")
for beta in (0.0, 0.05, 0.2, 0.5, 1.0, 3.0):
    pi = reinforce(beta=beta)
    ent = -(pi * pi.log()).sum().item()
    print("%4.2f   %10.3f        %8.3f       %8.3f      %6.3f" % (beta, (pi @ r_true).item(), kl(pi, pi_ref).item(), pi[3].item(), ent))
# closed-form optimum of E[r] - beta * KL: pi* is proportional to pi_ref * exp(r / beta)
print("closed form pi*(a) ~ pi_ref(a) * exp(r(a)/beta):")
for beta in (0.2, 0.5, 1.0):
    star = pi_ref * torch.exp(r_true / beta); star /= star.sum()
    print("  beta %.1f: expected reward %.3f, KL %.3f, P(best) %.3f" % (beta, (star @ r_true).item(), kl(star, pi_ref).item(), star[3].item()))

## 3. PPO's clipped objective: how far can one update move the policy?

In [ ]:
def ppo_step(clip, lr=0.5, batch=256, seed=0):
    g = torch.Generator().manual_seed(seed); theta = ref_logits.clone().requires_grad_(True)
    pi_old = F.softmax(theta.detach(), 0); a = torch.multinomial(pi_old, batch, replacement=True, generator=g)
    adv = r_true[a] - r_true[a].mean(); opt = torch.optim.SGD([theta], lr=lr)
    for _ in range(20):                                                       # many gradient steps on the SAME batch
        ratio = torch.exp(F.log_softmax(theta, 0)[a] - pi_old.log()[a])
        obj = torch.min(ratio * adv, ratio.clamp(1 - clip, 1 + clip) * adv) if clip else ratio * adv
        loss = -obj.mean(); opt.zero_grad(); loss.backward(); opt.step()
    return F.softmax(theta.detach(), 0)
print("\nPPO-style objective, 20 gradient steps on one batch:")
for clip in (None, 0.2, 0.1):
    pi = ppo_step(clip); print("  clip %-4s: expected reward %.3f, KL from old policy %.4f, max probability ratio vs old %.2f" % (clip, (pi @ r_true).item(), kl(pi, pi_ref).item(), (pi / pi_ref).max().item()))

## 4. DPO on preference PAIRS, no reward model

In [ ]:
def sample_pairs(n, seed=0):
    r = np.random.RandomState(seed); pairs = []
    for _ in range(n):
        i, j = r.choice(6, 2, replace=False); p_i = 1 / (1 + np.exp(-(r_true[i] - r_true[j]).item() * 4))       # Bradley-Terry preference
        pairs.append((i, j) if r.rand() < p_i else (j, i))
    return torch.tensor(pairs)
def dpo(pairs, beta=0.5, steps=6000, lr=0.5):
    theta = ref_logits.clone().requires_grad_(True); opt = torch.optim.SGD([theta], lr=lr); w, l = pairs[:, 0], pairs[:, 1]
    for _ in range(steps):
        lp = F.log_softmax(theta, 0); margin = beta * ((lp[w] - pi_ref.log()[w]) - (lp[l] - pi_ref.log()[l]))
        loss = -F.logsigmoid(margin).mean(); opt.zero_grad(); loss.backward(); opt.step()
    return F.softmax(theta.detach(), 0)
print("\nDPO (beta 0.5) from preference pairs only:")
# the pairs come from a Bradley-Terry model whose implicit reward is 4*r (preference prob = sigmoid(4*(r_i - r_j))), so the matching optimum uses 4*r/beta
star = pi_ref * torch.exp(4 * r_true / 0.5); star /= star.sum()
for n in (20, 100, 1000):
    pi = dpo(sample_pairs(n)); print("  %4d pairs: expected reward %.3f, KL %.3f, P(best) %.3f   (RLHF optimum: %.3f, %.3f, %.3f)" % (n, (pi @ r_true).item(), kl(pi, pi_ref).item(), pi[3].item(), (star @ r_true).item(), kl(star, pi_ref).item(), star[3].item()))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.